## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cstdlib>
using namespace std;

typedef long long LL;

const int MAXN = 1024;

vector<int> ans;
int o[MAXN], posi[MAXN];
int a, b, l, n;

inline void rf() {
    for (int i = 0; i < n; ++i)
        posi[o[i]] = i;
}

void doMagic() {
    ans.push_back(0);
    for (int i = 0; i < n; ++i) {
        if (o[i] == a) o[i] = b;
        else if (o[i] == b) o[i] = a;
    }
    rf();
}

void doAdd(int x) {
    if (x == 0) return;
    ans.push_back(x);
    for (int i = 0; i < n; ++i)
        o[i] = (o[i] + x) % n;
    rf();
}

void doXor(int x) {
    if (x == 0) return;
    ans.push_back(-x);
    for (int i = 0; i < n; ++i)
        o[i] = (o[i] ^ x);
    rf();
}

void calc(int av, int bv, int &pa, int &pb) {
    int delta = (bv - av + n - l + n) % n;
    pa = pb = 0;
    for (int stp = n / 2; stp >= 2 * l; stp /= 2) {
        if (delta >= stp) {
            delta -= stp;
            pb += stp / 2;
        } else {
            pa += stp / 2;
        }
    }
    pa += n / 2;
    pa += (av & (l - 1));
    pb += (av & (l - 1));
}

void doSwap(int c, int d) {
    if (c / l % 2 == d / l % 2) {
        int p;
        if (c / l % 2 == 0)
            p = (c & (l - 1)) + l;
        else
            p = (c & (l - 1));
        doSwap(c, p);
        doSwap(d, p);
        doSwap(c, p);
    } else {
        int pa, pb, pc, pd;
        calc(a, b, pa, pb);
        calc(c, d, pc, pd);
        doAdd((pc - c + n) % n);
        doXor(pc ^ pa);
        doAdd((a - pa + n) % n);
        doMagic();
        doAdd((pa - a + n) % n);
        doXor(pc ^ pa);
        doAdd((c - pc + n) % n);
    }
}

struct Perm {
    int a[MAXN];
    int n;
    vector<int> vec;

    bool calc() {
        // 检查 a[0..n-1] 是否是 0..n-1 的一个排列
        bool *f = new bool[n]();
        for (int i = 0; i < n; ++i) {
            if (a[i] < 0 || a[i] >= n || f[a[i]]) {
                delete[] f;
                return false;
            }
            f[a[i]] = true;
        }
        delete[] f;

        if (n == 1) return true;

        Perm b, c;
        for (int i = 0; i < n / 2; ++i) {
            b.a[i] = a[i * 2] / 2;
            c.a[i] = a[i * 2 + 1] / 2;
        }
        b.n = n / 2;
        c.n = n / 2;

        if (!b.calc() || !c.calc()) return false;

        if (a[0] % 2)
            vec.push_back(n == 2 ? 1 : -1);

        int tb = 0;
        for (int i : b.vec) {
            if (i > 0) {
                vec.push_back(-1);
                vec.push_back(1);
            } else {
                vec.push_back(i * 2);
                tb ^= -i * 2;
            }
        }
        if (tb) vec.push_back(-tb);

        int tc = 0;
        for (int i : c.vec) {
            if (i > 0) {
                vec.push_back(1);
                vec.push_back(-1);
            } else {
                vec.push_back(i * 2);
                tc ^= -i * 2;
            }
        }

        if ((tc & (n / 2)) != (tb & (n / 2))) {
            for (int i = 0; i < n / 4; ++i) {
                vec.push_back(-1);
                vec.push_back(1);
            }
        }

        if (tb >= (n / 2)) tb -= n / 2;
        if (tc >= (n / 2)) tc -= n / 2;
        if (tb != tc) return false;

        vector<int> tmp;
        for (int i : vec) {
            if (tmp.empty()) {
                tmp.push_back(i);
            } else {
                if (i < 0 && tmp.back() < 0) {
                    tmp.back() = -((-tmp.back()) ^ (-i));
                    if (tmp.back() == 0) tmp.pop_back();
                } else {
                    tmp.push_back(i);
                }
            }
        }
        swap(tmp, vec);
        return true;
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    cin >> n >> a >> b;
    for (int i = 0; i < n; ++i)
        cin >> o[i];

    rf();
    l = (a - b + n) % n;
    l = l & -l;
    if (l == 0) l = n;

    if (l > 1) {
        Perm p;
        p.n = l;
        for (int i = 0; i < n; ++i)
            p.a[i] = o[i] & (l - 1);

        if (!p.calc()) {
            cout << -1 << '\n';
            return 0;
        }
        for (int x : p.vec) {
            if (x > 0) doAdd(x);
            else doXor(-x);
        }
    }

    for (int i = 0; i < l; ++i) {
        vector<int> vec;
        for (int j = i; j < n; j += l)
            vec.push_back(o[j]);

        sort(vec.begin(), vec.end());
        int idx = 0;
        bool ok = true;
        for (int j = i; j < n; j += l) {
            if (vec[idx] != j) {
                ok = false;
                break;
            }
            ++idx;
        }
        if (!ok) {
            cout << -1 << '\n';
            return 0;
        }

        for (int j = i; j < n; j += l) {
            if (o[j] != j) {
                doSwap(j, o[j]);
            }
        }
    }

    // 最终验证
    for (int i = 0; i < n; ++i) {
        if (o[i] != i) {
            cout << -1 << '\n';
            return 0;
        }
    }

    cout << ans.size() << '\n';
    for (int v : ans) {
        if (v == 0) cout << "0\n";
        else if (v < 0) cout << "1 " << -v << '\n';
        else cout << "2 " << v << '\n';
    }

    return 0;
}

## B 长跑

In [ ]:
import java.io.*;
import java.util.*;

public class Main {
    static class FastReader {
        BufferedReader br;
        StringTokenizer st;
        public FastReader() { br = new BufferedReader(new InputStreamReader(System.in)); }
        String next() {
            while (st == null || !st.hasMoreElements()) {
                try { String line = br.readLine(); if (line == null) return null; st = new StringTokenizer(line); }
                catch (IOException e) { return null; }
            }
            return st.nextToken();
        }
        int nextInt() { return Integer.parseInt(next()); }
    }

    static class Node implements Comparable<Node> {
        int p, c;
        Node(int p, int c) { this.p = p; this.c = c; }
        public int compareTo(Node other) { return this.p - other.p; }
    }

    public static void main(String[] args) {
        FastReader fr = new FastReader();
        PrintWriter out = new PrintWriter(System.out);

        String nStr;
        while ((nStr = fr.next()) != null) {
            int n = Integer.parseInt(nStr);
            int l = fr.nextInt();
            int maxn = fr.nextInt();
            int s = fr.nextInt();

            List<Node> stations = new ArrayList<>();
            for (int i = 0; i < n; i++) {
                stations.add(new Node(fr.nextInt(), fr.nextInt()));
            }
            // 必须排序
            Collections.sort(stations);

            // 如果起点直接能到终点，不需要补给
            if (l <= maxn) {
                out.println("Yes");
                continue;
            }

            // minCost[i] 到达第 i 个补给站并补满体力所需的最少金币
            long[] minCost = new long[n];
            Arrays.fill(minCost, Long.MAX_VALUE);

            // 优先队列优化 DP: {cost, position}
            PriorityQueue<long[]> pq = new PriorityQueue<>(Comparator.comparingLong(a -> a[0]));

            // 初始化：从起点出发能直接到达的补给站，必须在那里买一次体力
            for (int i = 0; i < n; i++) {
                if (stations.get(i).p <= maxn) {
                    minCost[i] = stations.get(i).c;
                    pq.add(new long[]{minCost[i], stations.get(i).p});
                } else {
                    break; 
                }
            }

            long finalAns = Long.MAX_VALUE;

            // 如果起点出发一个站都到不了，且起点到不了终点
            if (pq.isEmpty()) {
                out.println("No");
                continue;
            }

            // 检查初始化里的站是否能直接到终点
            for (int i = 0; i < n; i++) {
                if (minCost[i] != Long.MAX_VALUE && (l - stations.get(i).p) <= maxn) {
                    finalAns = Math.min(finalAns, minCost[i]);
                }
            }

            // DP 转移
            for (int i = 0; i < n; i++) {
                Node cur = stations.get(i);
                
                // 维护 PQ，弹出当前体力够不着的站
                while (!pq.isEmpty() && (cur.p - (int)pq.peek()[1] > maxn)) {
                    pq.poll();
                }

                if (!pq.isEmpty()) {
                    long bestPrevCost = pq.peek()[0];
                    // 到达当前站并补给的花费
                    minCost[i] = Math.min(minCost[i], bestPrevCost + cur.c);
                    
                    // 如果从当前站补给后能到终点
                    if (l - cur.p <= maxn) {
                        finalAns = Math.min(finalAns, minCost[i]);
                    }
                    
                    // 将当前结果入队供后续使用
                    pq.add(new long[]{minCost[i], cur.p});
                }
            }

            if (finalAns <= (long)s) {
                out.println("Yes");
            } else {
                out.println("No");
            }
        }
        out.flush();
        out.close();
    }
}

## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ULL = unsigned long long;
const ULL B1 = 131, B2 = 137;
const ULL M1 = 1000000007, M2 = 1000000009;

class Hash {
    int len;
    vector<ULL> pre1, pre2, pow1, pow2;
public:
    Hash(const string& s) {
        len = s.size();
        pre1.assign(len + 1, 0);
        pre2.assign(len + 1, 0);
        pow1.assign(len + 1, 1);
        pow2.assign(len + 1, 1);
        for (int i = 0; i < len; ++i) {
            pre1[i + 1] = (pre1[i] * B1 + s[i]) % M1;
            pre2[i + 1] = (pre2[i] * B2 + s[i]) % M2;
            pow1[i + 1] = (pow1[i] * B1) % M1;
            pow2[i + 1] = (pow2[i] * B2) % M2;
        }
    }
    pair<ULL, ULL> get(int l, int r) { // 1-indexed inclusive
        if (l > r) return {0, 0};
        ULL v1 = (pre1[r] - pre1[l - 1] * pow1[r - l + 1] % M1 + M1) % M1;
        ULL v2 = (pre2[r] - pre2[l - 1] * pow2[r - l + 1] % M2 + M2) % M2;
        return {v1, v2};
    }
};

int LCP(Hash& ha, int la, int ra, Hash& hb, int lb, int rb, int) {
    int maxlen = min(ra - la + 1, rb - lb + 1);
    int lo = 1, hi = maxlen, best = 0;
    while (lo <= hi) {
        int mid = (lo + hi) >> 1;
        if (ha.get(la, la + mid - 1) == hb.get(lb, lb + mid - 1)) {
            best = mid;
            lo = mid + 1;
        } else {
            hi = mid - 1;
        }
    }
    return best;
}

vector<pair<int, int>> manacher(const string& str) {
    string t = "#";
    for (char ch : str) t += ch, t += "#";
    int m = t.size();
    vector<int> d(m, 0);
    int l = 0, r = -1;
    for (int i = 0; i < m; ++i) {
        int k = (i > r) ? 1 : min(d[l + r - i], r - i + 1);
        while (i - k >= 0 && i + k < m && t[i - k] == t[i + k]) ++k;
        d[i] = k--;
        if (i + k > r) {
            l = i - k;
            r = i + k;
        }
    }
    vector<pair<int, int>> intervals;
    for (int i = 0; i < m; ++i) {
        int radius = d[i] - 1;
        if (radius <= 0) continue;
        int L = (i - radius + 2) >> 1;
        int R = (i + radius) >> 1;
        intervals.emplace_back(L, R);
    }
    return intervals;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;
    string A, B;
    cin >> A >> B;
    string revA = A;
    reverse(revA.begin(), revA.end());

    Hash hrev(revA), hB(B);

    auto getCommon = [&](int posA, int posB) -> int {
        if (posA < 1 || posA > n || posB < 1 || posB > n) return 0;
        return LCP(hrev, n - posA + 1, n, hB, posB, n, n);
    };

    vector<pair<int, int>> palA = manacher(A);
    vector<pair<int, int>> palB = manacher(B);

    int answer = 0;
    for (auto& seg : palA) answer = max(answer, seg.second - seg.first + 1);
    for (auto& seg : palB) answer = max(answer, seg.second - seg.first + 1);

    for (int i = 1; i <= n; ++i)
        answer = max(answer, 2 * getCommon(i, i));

    for (auto& seg : palA) {
        int curLen = seg.second - seg.first + 1;
        if (seg.first > 1)
            answer = max(answer, curLen + 2 * getCommon(seg.first - 1, seg.second));
    }
    for (auto& seg : palB) {
        int curLen = seg.second - seg.first + 1;
        if (seg.second < n)
            answer = max(answer, curLen + 2 * getCommon(seg.first, seg.second + 1));
    }

    cout << answer << '\n';
    return 0;
}

## D 优惠券

In [ ]:
#include <stdio.h>
#include <string.h>

#define MAXN 555555

char op[MAXN];
int last_pos[MAXN];
int id[MAXN];
int bit[MAXN];
int n;

int lowbit(int x) {
    return x & -x;
}

void add(int x, int v) {
    while (x <= n) {
        bit[x] += v;
        x += lowbit(x);
    }
}

int sum(int x) {
    int res = 0;
    while (x > 0) {
        res += bit[x];
        x -= lowbit(x);
    }
    return res;
}

int kth(int k) {
    int pos = 0;
    int step = 1;

    while ((step << 1) <= n) {
        step <<= 1;
    }

    while (step > 0) {
        int next = pos + step;
        if (next <= n && bit[next] < k) {
            pos = next;
            k -= bit[next];
        }
        step >>= 1;
    }

    return pos + 1;
}

int find_first_ge(int x) {
    if (x < 1) {
        x = 1;
    }

    int before = sum(x - 1);
    int total = sum(n);

    if (before == total) {
        return -1;
    }

    return kth(before + 1);
}

int main() {
    while (scanf("%d", &n) != EOF) {
        memset(last_pos, 0, sizeof(last_pos));
        memset(bit, 0, sizeof(bit));

        int ans = -1;
        int ok = 1;

        for (int i = 1; i <= n; i++) {
            char s[5];
            scanf("%s", s);
            op[i] = s[0];

            if (op[i] != '?') {
                scanf("%d", &id[i]);
            }

            if (!ok) {
                continue;
            }

            if (op[i] == '?') {
                add(i, 1);
            } else {
                int x = id[i];

                if (last_pos[x] != 0) {
                    int pre = last_pos[x];

                    if (op[pre] == op[i]) {
                        int p = find_first_ge(pre);

                        if (p == -1) {
                            ans = i;
                            ok = 0;
                        } else {
                            add(p, -1);
                        }
                    }

                    last_pos[x] = i;
                } else {
                    if (op[i] == 'O') {
                        int p = find_first_ge(1);

                        if (p == -1) {
                            ans = i;
                            ok = 0;
                        } else {
                            add(p, -1);
                        }
                    }

                    last_pos[x] = i;
                }
            }
        }

        printf("%d\n", ans);
    }

    return 0;
}

## E 任意点

In [ ]:
#include <iostream>
#include <vector>
#include <unordered_map>
#include <unordered_set>
#include <algorithm>
using namespace std;

struct DSU {
    vector<int> parent;
    DSU(int n) {
        parent.resize(n);
        for (int i = 0; i < n; ++i) parent[i] = i;
    }
    int find(int x) {
        if (parent[x] != x) parent[x] = find(parent[x]);
        return parent[x];
    }
    void unite(int x, int y) {
        x = find(x), y = find(y);
        if (x != y) parent[y] = x;
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;
    vector<pair<int, int>> points(n);
    unordered_set<int> xs, ys;
    unordered_map<int, int> x_id, y_id;

    // 1. 读取点，收集x、y坐标
    for (int i = 0; i < n; ++i) {
        int x, y;
        cin >> x >> y;
        points[i] = {x, y};
        xs.insert(x);
        ys.insert(y);
    }

    // 2. 坐标离散化：给x、y分配唯一id
    int cnt_x = 0, cnt_y = 0;
    for (int x : xs) x_id[x] = cnt_x++;
    for (int y : ys) y_id[y] = cnt_x + cnt_y++; // y的id从cnt_x开始，避免和x冲突

    // 3. 初始化并查集：总节点数 = x数量 + y数量
    DSU dsu(cnt_x + cnt_y);
    for (auto& p : points) {
        int x = p.first, y = p.second;
        int u = x_id[x], v = y_id[y];
        dsu.unite(u, v);
    }

    // 4. 统计连通块数量
    unordered_set<int> roots;
    for (int i = 0; i < cnt_x + cnt_y; ++i) {
        roots.insert(dsu.find(i));
    }
    int C = roots.size();

    // 5. 计算答案：C-1（n=1时C=1，结果0，符合要求）
    cout << (C - 1) << endl;

    return 0;
}

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Piece {
    string s;
    int offset;
    vector<int> pi;
};

struct Segment {
    string raw;
    int len;
    vector<Piece> pieces;
};

vector<int> getPi(const string &p) {
    int n = p.size();
    vector<int> pi(n, 0);

    for (int i = 1; i < n; i++) {
        int j = pi[i - 1];

        while (j > 0 && p[i] != p[j]) {
            j = pi[j - 1];
        }

        if (p[i] == p[j]) {
            j++;
        }

        pi[i] = j;
    }

    return pi;
}

Segment buildSegment(const string &s) {
    Segment seg;
    seg.raw = s;
    seg.len = (int)s.size();

    int n = s.size();

    for (int i = 0; i < n; ) {
        if (s[i] == '?') {
            i++;
        } else {
            int j = i;
            while (j < n && s[j] != '?') {
                j++;
            }

            Piece piece;
            piece.s = s.substr(i, j - i);
            piece.offset = i;
            piece.pi = getPi(piece.s);

            seg.pieces.push_back(piece);

            i = j;
        }
    }

    return seg;
}

bool fixedMatch(const Segment &seg, const string &text, int start) {
    int m = text.size();

    if (start < 0 || start + seg.len > m) {
        return false;
    }

    for (int i = 0; i < seg.len; i++) {
        if (seg.raw[i] != '?' && seg.raw[i] != text[start + i]) {
            return false;
        }
    }

    return true;
}

void kmpAdd(
    const Piece &piece,
    const string &text,
    vector<int> &cnt,
    int beginStart,
    int maxStart
) {
    const string &p = piece.s;
    const vector<int> &pi = piece.pi;

    int m = text.size();
    int len = p.size();
    int j = 0;

    for (int i = 0; i < m; i++) {
        while (j > 0 && text[i] != p[j]) {
            j = pi[j - 1];
        }

        if (text[i] == p[j]) {
            j++;
        }

        if (j == len) {
            int occurPos = i - len + 1;
            int segStart = occurPos - piece.offset;

            if (segStart >= beginStart && segStart <= maxStart) {
                cnt[segStart - beginStart]++;
            }

            j = pi[j - 1];
        }
    }
}

int findEarliest(const Segment &seg, const string &text, int beginStart, int endLimit) {
    int maxStart = endLimit - seg.len;

    if (maxStart < beginStart) {
        return -1;
    }

    if (seg.pieces.empty()) {
        return beginStart;
    }

    vector<int> cnt(maxStart - beginStart + 1, 0);

    for (int i = 0; i < (int)seg.pieces.size(); i++) {
        kmpAdd(seg.pieces[i], text, cnt, beginStart, maxStart);
    }

    int need = seg.pieces.size();

    for (int i = 0; i < (int)cnt.size(); i++) {
        if (cnt[i] == need) {
            return beginStart + i;
        }
    }

    return -1;
}

bool normalMatch(const string &pattern, const string &text) {
    if (pattern.size() != text.size()) {
        return false;
    }

    for (int i = 0; i < (int)pattern.size(); i++) {
        if (pattern[i] != '?' && pattern[i] != text[i]) {
            return false;
        }
    }

    return true;
}

bool matchWildcard(const vector<Segment> &segs, const string &pattern, const string &text) {
    bool hasStar = false;

    for (char ch : pattern) {
        if (ch == '*') {
            hasStar = true;
            break;
        }
    }

    if (!hasStar) {
        return normalMatch(pattern, text);
    }

    bool leftAnchor = pattern[0] != '*';
    bool rightAnchor = pattern.back() != '*';

    int m = text.size();
    int pos = 0;
    int endLimit = m;

    int l = 0;
    int r = (int)segs.size() - 1;

    if (leftAnchor) {
        if (l > r) {
            return false;
        }

        if (!fixedMatch(segs[l], text, 0)) {
            return false;
        }

        pos = segs[l].len;
        l++;
    }

    if (rightAnchor) {
        if (l > r) {
            return pos <= m;
        }

        int start = m - segs[r].len;

        if (!fixedMatch(segs[r], text, start)) {
            return false;
        }

        endLimit = start;
        r--;
    }

    if (pos > endLimit) {
        return false;
    }

    for (int i = l; i <= r; i++) {
        int start = findEarliest(segs[i], text, pos, endLimit);

        if (start == -1) {
            return false;
        }

        pos = start + segs[i].len;

        if (pos > endLimit) {
            return false;
        }
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;

    if (!(cin >> pattern)) {
        return 0;
    }

    int n;

    if (!(cin >> n)) {
        return 0;
    }

    vector<Segment> segs;
    string now = "";

    for (char ch : pattern) {
        if (ch == '*') {
            if (!now.empty()) {
                segs.push_back(buildSegment(now));
                now.clear();
            }
        } else {
            now.push_back(ch);
        }
    }

    if (!now.empty()) {
        segs.push_back(buildSegment(now));
    }

    while (n--) {
        string filename;
        cin >> filename;

        if (matchWildcard(segs, pattern, filename)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;

typedef long long ll;

vector<pair<int, int>> priorityMove;

int getPeg(char c) {
    if (c == 'A') return 0;
    if (c == 'B') return 1;
    return 2;
}

ll simulateSmall(int n) {
    vector<int> pos(n + 1, 0); 
    int lastDisk = -1;
    ll step = 0;

    while (true) {
        bool finish = true;
        for (int i = 2; i <= n; i++) {
            if (pos[i] != pos[1]) {
                finish = false;
                break;
            }
        }

        if (step > 0 && finish && pos[1] != 0) {
            return step;
        }

        int top[3];
        top[0] = top[1] = top[2] = n + 1;

        for (int disk = 1; disk <= n; disk++) {
            int p = pos[disk];
            top[p] = min(top[p], disk);
        }

        for (auto mv : priorityMove) {
            int from = mv.first;
            int to = mv.second;

            if (top[from] == n + 1) {
                continue;
            }

            int disk = top[from];

            if (disk == lastDisk) {
                continue;
            }

            if (top[to] == n + 1 || disk < top[to]) {
                pos[disk] = to;
                lastDisk = disk;
                step++;
                break;
            }
        }
    }
}

ll fastPow(ll a, int b) {
    ll res = 1;
    while (b--) {
        res *= a;
    }
    return res;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    priorityMove.clear();

    for (int i = 0; i < 6; i++) {
        string s;
        cin >> s;

        int from = getPeg(s[0]);
        int to = getPeg(s[1]);

        priorityMove.push_back({from, to});
    }

    if (n <= 3) {
        cout << simulateSmall(n) << '\n';
        return 0;
    }

    ll t2 = simulateSmall(2);
    ll t3 = simulateSmall(3);

    ll ans;

    if (t2 == 5) {
        ans = 2 * fastPow(3, n - 1) - 1;
    } else {
        if (t3 == 7) {
            ans = fastPow(2, n) - 1;
        } else {
            ans = fastPow(3, n - 1);
        }
    }

    cout << ans << '\n';

    return 0;
}

## H 马步距离

In [ ]:
#include <bits/stdc++.h>
using namespace std;

long long knightDistance(long long dx, long long dy) {
    dx = llabs(dx);
    dy = llabs(dy);

    if (dx < dy) {
        swap(dx, dy);
    }

    if (dx == 0 && dy == 0) {
        return 0;
    }

    if (dx == 1 && dy == 0) {
        return 3;
    }

    if (dx == 2 && dy == 2) {
        return 4;
    }

    long long ans = max((dx + 1) / 2, (dx + dy + 2) / 3);

    if ((ans + dx + dy) % 2 != 0) {
        ans++;
    }

    return ans;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    long long xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    long long dx = xs - xp;
    long long dy = ys - yp;

    cout << knightDistance(dx, dy) << '\n';

    return 0;
}

## I 直方图最大矩形

In [ ]:
#include <bits/stdc++.h>
using namespace std;

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();

        if (n == 0) {
            return 0;
        }

        stack<int> st;
        long long ans = 0;

        for (int i = 0; i <= n; i++) {
            int curHeight;

            if (i == n) {
                curHeight = 0;
            } else {
                curHeight = heights[i];
            }

            while (!st.empty() && heights[st.top()] > curHeight) {
                int h = heights[st.top()];
                st.pop();

                int left;
                if (st.empty()) {
                    left = -1;
                } else {
                    left = st.top();
                }

                int width = i - left - 1;
                long long area = 1LL * h * width;

                if (area > ans) {
                    ans = area;
                }
            }

            st.push(i);
        }

        return (int)ans;
    }
};

## J 消防局的设立

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<int> fa(n + 1, 0);
    vector<vector<int>> child(n + 1);

    fa[1] = 0;

    for (int i = 2; i <= n; i++) {
        cin >> fa[i];
        child[fa[i]].push_back(i);
    }

    vector<int> depth(n + 1, 0);
    vector<int> order;

    queue<int> q;
    q.push(1);

    while (!q.empty()) {
        int u = q.front();
        q.pop();

        order.push_back(u);

        for (int v : child[u]) {
            depth[v] = depth[u] + 1;
            q.push(v);
        }
    }

    sort(order.begin(), order.end(), [&](int x, int y) {
        return depth[x] > depth[y];
    });

    vector<int> hasStation(n + 1, 0);
    vector<int> childStation(n + 1, 0);
    vector<int> grandChildStation(n + 1, 0);

    auto covered = [&](int u) {
        int p = fa[u];
        int gp = p ? fa[p] : 0;

        if (hasStation[u]) return true;
        if (p && hasStation[p]) return true;
        if (gp && hasStation[gp]) return true;

        if (childStation[u] > 0) return true;
        if (p && childStation[p] > 0) return true;
        if (grandChildStation[u] > 0) return true;

        return false;
    };

    auto buildStation = [&](int u) {
        if (hasStation[u]) return;

        hasStation[u] = 1;

        int p = fa[u];
        int gp = p ? fa[p] : 0;

        if (p) {
            childStation[p]++;
        }

        if (gp) {
            grandChildStation[gp]++;
        }
    };

    int ans = 0;

    for (int u : order) {
        if (!covered(u)) {
            int p = fa[u];
            int gp = p ? fa[p] : 0;

            int pos;
            if (gp) {
                pos = gp;
            } else {
                pos = 1;
            }

            buildStation(pos);
            ans++;
        }
    }

    cout << ans << '\n';

    return 0;
}